# 06 — Chia máy trạm theo 4 bệnh viện  ·  Bước 6

**Vì sao.** Các kịch bản ở notebook 02 (Dirichlet) là bệnh viện *giả*: chỉ lệch nhau về loại bệnh. Bước 3
và 5 cho thấy bốn nguồn thật của HAM10000 còn lệch nhau về *cách ảnh trông ra*. Đó là chữ ký của máy
chụp, và chuẩn hoá màu chỉ xoá được một phần. Notebook này tạo kịch bản `nguon`: mỗi nguồn là một máy
trạm.

Làm ba việc:

1. Thêm hai cột vào `chia_tap.csv`:
   - `nguon`: nguồn của mỗi ảnh train và val;
   - `may_nguon`: số thứ tự máy trạm (0–3), chỉ ở các dòng train.
2. Kiểm tra: không tổn thương nào bị xé ra hai máy; số ảnh và độ lệch nhãn của từng máy.
3. Đếm val theo nguồn. Sau này mình sẽ chấm điểm **riêng từng bệnh viện** trên val, vì test chính thức
   không ghi nguồn.

Trước khi ghi đè, notebook lưu một bản sao `chia_tap_truoc_06.csv`. Các cột cũ không bị đụng tới.

In [1]:
import shutil
import numpy as np, pandas as pd
from scipy.spatial.distance import jensenshannon
from du_lieu import RA, LOP

NGUON = ["vidir_molemax", "vidir_modern", "rosendahl", "vienna_dias"]
cs = pd.read_csv(RA / "chia_tap.csv")
ng = pd.read_csv(RA / "nguon.csv")[["image", "nguon"]]           # từ notebook 03: nguồn đã gán theo tổn thương
if not (RA / "chia_tap_truoc_06.csv").exists():
    shutil.copy(RA / "chia_tap.csv", RA / "chia_tap_truoc_06.csv")
cs = cs.drop(columns=[c for c in ("nguon", "may_nguon") if c in cs.columns]).merge(ng, on="image", how="left")
la_tv = cs.phan.isin(["train", "val"])
assert cs.loc[la_tv, "nguon"].notna().all(), "Có ảnh train/val chưa có nguồn"
cs["may_nguon"] = np.where(cs.phan == "train", cs.nguon.map({s: i for i, s in enumerate(NGUON)}), np.nan)
print(cs.groupby("phan").nguon.apply(lambda s: s.notna().sum()).to_string())

phan
test                 0
train             8583
val               1432
val_chinh_thuc       0


## Kiểm tra

Hai điều phải đúng. Thứ nhất, không tổn thương nào nằm ở hai máy trạm (giống nguyên tắc ở notebook 02).
Thứ hai, độ lệch nhãn khớp với con số đã đo ở notebook 03: JS trung bình khoảng 0,214.

In [2]:
tr = cs[cs.phan == "train"]
xe = (tr.groupby("lesion_id").may_nguon.nunique() > 1).sum()
print(f"Tổn thương nằm ở >1 máy trạm: {xe}   (phải bằng 0)")
assert xe == 0

chung = tr.lop.value_counts(normalize=True).reindex(LOP).fillna(0).values
b = pd.crosstab(tr.may_nguon.astype(int), tr.lop).reindex(columns=LOP, fill_value=0)
bang = pd.DataFrame({"nguon": NGUON, "so_anh_train": b.sum(axis=1).values,
                     "JS": [jensenshannon(r / r.sum(), chung) for r in b.values]})
bang = pd.concat([bang, (b.div(b.sum(axis=1), axis=0) * 100).round(1).reset_index(drop=True)], axis=1)
display(bang.round(3))
print(f"JS trung bình: {bang.JS.mean():.3f}   (notebook 03: 0.214)")

Tổn thương nằm ở >1 máy trạm: 0   (phải bằng 0)


,nguon,so_anh_train,JS,MEL,NV,BCC,AKIEC,BKL,DF,VASC
0,vidir_molemax,3388,0.286,0.4,94.3,0.1,0.0,3.2,0.7,1.3
1,vidir_modern,2904,0.125,20.0,54.6,6.4,0.8,14.2,1.5,2.4
2,rosendahl,1926,0.252,15.5,35.3,12.9,13.3,21.5,1.4,0.2
3,vienna_dias,365,0.194,15.9,78.6,1.1,0.0,2.5,1.1,0.8


JS trung bình: 0.214   (notebook 03: 0.214)


In [3]:
va = cs[cs.phan == "val"]
display(pd.crosstab(va.nguon, va.lop).reindex(index=NGUON, columns=LOP, fill_value=0).assign(tong=lambda d: d.sum(axis=1)))
print("Lưu ý: vienna_dias có rất ít ảnh val, nên điểm riêng của bệnh viện này sẽ nhiễu.")

lop,MEL,NV,BCC,AKIEC,BKL,DF,VASC,tong
nguon,,,,,,,,
vidir_molemax,2,525,0,0,17,6,9,559
vidir_modern,105,246,25,8,63,8,11,466
rosendahl,43,124,48,39,76,3,0,333
vienna_dias,9,63,1,0,1,0,0,74


Lưu ý: vienna_dias có rất ít ảnh val, nên điểm riêng của bệnh viện này sẽ nhiễu.


In [4]:
cs.to_csv(RA / "chia_tap.csv", index=False)
print(f"Đã ghi {RA / 'chia_tap.csv'} — thêm cột nguon, may_nguon. Bản cũ: chia_tap_truoc_06.csv")
print("===== TÓM TẮT NOTEBOOK 06 =====")
print(bang[["nguon", "so_anh_train", "JS"]].round(3).to_string(index=False))
print("Val theo nguồn:", va.nguon.value_counts().reindex(NGUON).to_dict())

Đã ghi /home/deep/ham_fl/ketqua/chia_tap.csv — thêm cột nguon, may_nguon. Bản cũ: chia_tap_truoc_06.csv
===== TÓM TẮT NOTEBOOK 06 =====
        nguon  so_anh_train    JS
vidir_molemax          3388 0.286
 vidir_modern          2904 0.125
    rosendahl          1926 0.252
  vienna_dias           365 0.194
Val theo nguồn: {'vidir_molemax': 559, 'vidir_modern': 466, 'rosendahl': 333, 'vienna_dias': 74}
